### Import librerie


In [ ]:
import os
import argparse
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import tensorflow as tf
import keras
import xgboost as xgb
from keras import layers
import json
import matplotlib.pyplot as plt

### Data Utilities


In [ ]:
# Costanti
HYPO = 70.0
HYPER = 180.0
L_BOUND = 40.0
U_BOUND = 400.0

In [ ]:
def load_splits(splits_dir="data/split_sets"):
    """Carica gli split e i metadati dai relativi file.
    Args:
        splits_dir (str): Directory contenente gli split set (default: 'data/split_sets')
    Returns:
        tuple: (train_set, val_set, test_set, X_cols, y_cols)
    """
    datasets = []
    for name in ["train", "val", "test"]:
        df = pd.read_parquet(f"{splits_dir}/{name}_set.parquet")
        datasets.append(df)

    with open(f"{splits_dir}/metadata.json", "r") as f:
        metadata = json.load(f)

    return tuple(datasets + [metadata["X_cols"], metadata["y_cols"]])

In [ ]:
def rescale_data(df, rescale_cols):
    """Effettua il rescale dei dati back al loro intervallo originario.
    Args:
        df (pd.DataFrame): DataFrame con i dati da riscalare
        rescale_cols (list): List dei nomi delle colonne da riscalare
    Returns:
        pd.DataFrame: Dataframe con le colonne scelte riscalate
    """
    df = df.copy()
    for col in rescale_cols:
        df[col] = ((df[col] + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
    return df

In [ ]:
def calculate_metrics(df):
    """Calcola le metriche per ciascun paziente in un sottoinsieme.
    Args:
        df (pd.DataFrame): Dataframe con le colonne 'Patient_ID', 'target', e 'y_pred'
    Returns:
        tuple: (samples, maes, mapes, rmses) - numero di samples e lista delle metriche ottenute
    """
    samples = 0
    maes, mapes, rmses = [], [], []

    for patient_id in df["Patient_ID"].unique():
        patient_data = df[df["Patient_ID"] == patient_id]
        if patient_data.empty:
            continue

        samples += len(patient_data)
        maes.append(mean_absolute_error(patient_data["target"], patient_data["y_pred"]))
        mapes.append(
            mean_absolute_percentage_error(
                patient_data["target"], patient_data["y_pred"]
            )
            * 100
        )
        rmses.append(
            root_mean_squared_error(patient_data["target"], patient_data["y_pred"])
        )

    return samples, maes, mapes, rmses

In [ ]:
def print_results(df):
    """Stampa i risultati delle valutazioni cumulative e per condizione glicemica.
    Args:
        df (pd.DataFrame): DataFrame con valori inferiti e di riferimento
    """

    def print_metrics(title, samples, maes, mapes, rmses):
        """Stampa le metriche formattate per una specifica condizione."""
        if title != "Cumulative":
            print("~" * 10)
        print(title)
        print(f"Samples: {samples}")
        if maes:  # Stampa solo se abbiamo dei dati
            print(f"MAE: {np.mean(maes):.2f}({np.std(maes):.2f})")
            print(f"MAPE: {np.mean(mapes):.2f}({np.std(mapes):.2f})")
            print(f"RMSE: {np.mean(rmses):.2f}({np.std(rmses):.2f})")

    # Overall results
    samples, maes, mapes, rmses = calculate_metrics(df)
    print_metrics("Cumulative", samples, maes, mapes, rmses)

    # Results by condition
    for condition in ["Normal", "Hyper", "Hypo"]:
        condition_df = df[df["bgClass"] == condition]
        samples, maes, mapes, rmses = calculate_metrics(condition_df)
        print_metrics(condition, samples, maes, mapes, rmses)

### Dnn Utilities


In [ ]:
def create_gru_model():
    """Costruisci il modello GRU"""
    return keras.Sequential(
        [
            layers.Input(shape=(8, 1)),
            layers.GRU(86, return_sequences=True),
            layers.Dropout(0.2),
            layers.GRU(86, return_sequences=False),
            layers.Dense(1),
        ],
        name="GRU_Model",
    )

In [ ]:
def predict_in_batches(model, data, model_type, batch_size=256):
    """Esegui le predizione ed effettua il reshaping automatico dei dati"""
    if model_type not in ["mlp", "lstm", "gru"]:
        raise ValueError(f"Model type must be one of {['mlp', 'lstm', 'gru']}")

    def _reshape_for_rnn(X):
        """Esegui il reshape de i dati per i modelli RNN"""
        return X.reshape(X.shape[0], X.shape[1], 1)

    # Prepare data based on model type
    if model_type in ["lstm", "gru"]:
        data_reshaped = _reshape_for_rnn(data.values)
    else:
        data_reshaped = data.values

    return model.predict(data_reshaped, batch_size=batch_size, verbose=0)

In [ ]:
def print_model_summary(model):
    """Stampa l'architettura del modello ed il conteggio dei parametri"""
    print("\nModel Architecture:")
    model.summary()
    print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
def create_callbacks(**kwargs):
    """Crea i callbacks per il training con i valori di default"""
    defaults = {
        "early_stopping_patience": 5,
        "early_stopping_min_delta": 1e-5,
        "lr_scheduler": True,
        "lr_reduction_factor": 0.1,
        "lr_scheduler_patience": 3,
        "min_learning_rate": 1e-6,
        "monitor": "val_loss",
        "verbose": 1,
    }

    config = {**defaults, **kwargs}

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor=config["monitor"],
            patience=config["early_stopping_patience"],
            min_delta=config["early_stopping_min_delta"],
            verbose=config["verbose"],
            restore_best_weights=True,
        )
    ]

    if config["lr_scheduler"]:
        callbacks.append(
            keras.callbacks.ReduceLROnPlateau(
                monitor=config["monitor"],
                factor=config["lr_reduction_factor"],
                patience=config["lr_scheduler_patience"],
                min_lr=config["min_learning_rate"],
                verbose=config["verbose"],
            )
        )

    return callbacks

### Clarke Error Grid Utilities


In [ ]:
def _calculate_clarke_zones(ref_values, pred_values):
    """
    Calculate Clarke Error Grid zone assignments for each point
    Args:
        ref_values (numpy.ndarray): Reference glucose values
        pred_values (numpy.ndarray): Predicted glucose values
    Returns:
        list: List of 5 integers representing counts in zones A, B, C, D, E
    """
    zone = [0] * 5

    for i in range(len(ref_values)):
        ref_val = ref_values[i]
        pred_val = pred_values[i]

        # Zone A: Clinically accurate values
        if (ref_val <= 70 and pred_val <= 70) or (
            pred_val <= 1.2 * ref_val and pred_val >= 0.8 * ref_val
        ):
            zone[0] += 1  # Zone A

        # Zone E: Erroneous values (most dangerous)
        elif (ref_val >= 180 and pred_val <= 70) or (ref_val <= 70 and pred_val >= 180):
            zone[4] += 1  # Zone E

        # Zone C: Overcorrection values
        elif ((ref_val >= 70 and ref_val <= 290) and pred_val >= ref_val + 110) or (
            (ref_val >= 130 and ref_val <= 180)
            and (pred_val <= (7 / 5) * ref_val - 182)
        ):
            zone[2] += 1  # Zone C

        # Zone D: Dangerous failure to detect values
        elif (
            (ref_val >= 240 and (pred_val >= 70 and pred_val <= 180))
            or (ref_val <= 175 / 3 and pred_val <= 180 and pred_val >= 70)
            or (
                (ref_val >= 175 / 3 and ref_val <= 70) and pred_val >= (6 / 5) * ref_val
            )
        ):
            zone[3] += 1  # Zone D

        # Zone B: Benign errors
        else:
            zone[1] += 1  # Zone B

    return zone

In [ ]:
def _add_clarke_zone_boundaries():
    """Add Clarke Error Grid zone boundary lines to the current plot"""
    zone_line_color = "black"
    zone_line_width = 1.5

    # Zone boundary lines according to Clarke Error Grid specification
    plt.plot([0, 175 / 3], [70, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot(
        [175 / 3, 400 / 1.2],
        [70, 400],
        "-",
        c=zone_line_color,
        linewidth=zone_line_width,
    )
    plt.plot([70, 70], [84, 400], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([0, 70], [180, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 290], [180, 400], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 70], [0, 56], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([70, 400], [56, 320], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([180, 180], [0, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([180, 400], [70, 70], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([240, 240], [70, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([240, 400], [180, 180], "-", c=zone_line_color, linewidth=zone_line_width)
    plt.plot([130, 180], [0, 70], "-", c=zone_line_color, linewidth=zone_line_width)

In [ ]:
def _add_clarke_zone_labels():
    """Add zone labels (A, B, C, D, E) to the current plot"""
    zone_font_size = 15
    zone_font_weight = "bold"

    # Zone A
    plt.text(
        30, 15, "A", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone B (appears in two regions)
    plt.text(
        370, 220, "B", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        290, 370, "B", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone C (appears in two regions)
    plt.text(
        160, 370, "C", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        160, 15, "C", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone D (appears in two regions)
    plt.text(
        30, 140, "D", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        370, 90, "D", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

    # Zone E (appears in two regions)
    plt.text(
        30, 370, "E", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )
    plt.text(
        370, 15, "E", fontsize=zone_font_size, fontweight=zone_font_weight, ha="center"
    )

In [ ]:
def _create_clarke_error_grid_plot(ref_values, pred_values, title_string, save_path):
    """
    Create and save Clarke Error Grid visualization
    Args:
        ref_values (array-like): Valori glicemici di riferimento (mg/dl)
        pred_values (array-like): Valori glicemici inferiti (mg/dl)
        title_string (str): Titolo per il plot
        save_path (str): Path dove salvare il plot. Se None, il plot non viene salvato
    """

    # plt.figure(figsize=(8, 8), dpi=300)

    plt.figure(dpi=300)

    # Plot data points
    plt.scatter(
        ref_values,
        pred_values,
        marker="o",
        color="steelblue",
        s=12,
        # alpha=0.6,
        edgecolors="black",
        linewidth=0.1,
    )

    # Set labels and title

    # plt.title(title_string + " Clarke Error Grid", fontsize=16, fontweight="bold")

    plt.xlabel("Glicemia di riferimento (mg/dL)", fontsize=14)
    plt.ylabel("Glicemia inferita (mg/dL)", fontsize=14)

    # Add perfect prediction line
    plt.plot(
        [0, 400],
        [0, 400],
        ":",
        c="gray",
        linewidth=2,
        alpha=0.7,
        label="Perfect prediction",
    )

    # Add zone boundaries
    _add_clarke_zone_boundaries()

    # Add zone labels
    _add_clarke_zone_labels()

    # Configure plot appearance
    plt.xlim([0, 400])
    plt.ylim([0, 400])
    plt.grid(True, alpha=0.3, linestyle="--")
    plt.gca().set_aspect("equal")
    plt.tight_layout()

    # Save plot
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

In [ ]:
def clarke_error_grid_analysis(ref_values, pred_values, title_string, save_path=None):
    """
    Esegui un'analisi tramite la CEG e restituisci le statistiche per ogni zona
    Args:
        ref_values (array-like): Valori glicemici di riferimento (mg/dl)
        pred_values (array-like): Valori glicemici inferiti (mg/dl)
        title_string (str): Titolo per il plot
        save_path (str): Path dove salvare il plot. Se None, il plot non viene salvato
    Returns:
        dict: Dizionario contenente le statistiche per ogni zona:
        - zone_counts: Count dei punti per ciascuna zona (A, B, C, D, E)
        - zone_percentages: Percentuale dei punti in ciascuna zona
        - total_points: Numero totale di punti validi (che ricadono nella griglia)
        - clinically_acceptable: Percentuale nelle zone A+B
        - clinically_dangerous: Percentuale nelle zone D+E
    """
    # Convert to numpy arrays
    ref_values = np.array(ref_values)
    pred_values = np.array(pred_values)

    # Filter out invalid values
    valid_mask = (
        ~np.isnan(ref_values)
        & ~np.isnan(pred_values)
        & (ref_values >= 0)
        & (pred_values >= 0)
        & (ref_values <= 400)
        & (pred_values <= 400)
    )

    ref_values = ref_values[valid_mask]
    pred_values = pred_values[valid_mask]

    # Calculate zone statistics
    zone_counts = _calculate_clarke_zones(ref_values, pred_values)

    total_points = sum(zone_counts)
    zone_percentages = [count / total_points * 100 for count in zone_counts]

    stats = {
        "zone_counts": dict(zip(["A", "B", "C", "D", "E"], zone_counts)),
        "zone_percentages": dict(zip(["A", "B", "C", "D", "E"], zone_percentages)),
        "total_points": total_points,
        "clinically_acceptable": (zone_counts[0] + zone_counts[1]) / total_points * 100,
        "clinically_dangerous": (zone_counts[3] + zone_counts[4]) / total_points * 100,
    }

    # Create plot if save_path is provided
    if save_path:
        _create_clarke_error_grid_plot(ref_values, pred_values, title_string, save_path)
        print(f"Clarke Error Grid saved to: {save_path}")

    return stats

### General vs Personalized Utilities


In [ ]:
def setup_environment(args):
    """Setup delle directory e del random seed"""
    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs(args.models_dir, exist_ok=True)
    os.makedirs(args.scores_dir, exist_ok=True)
    os.makedirs(args.plots_dir, exist_ok=True)

    print(f"Output directory: {args.output_dir}")

In [ ]:
def set_seeds(seed):
    """Imposta il random seed per la riproducibilità in TensorFlow/Keras."""
    # Set NumPy seed
    np.random.seed(seed)
    # Set TensorFlow seeds
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

In [ ]:
def identify_extreme_patients(gru_results_file):
    """Identifica pazienti con migliore e peggiore performance dal GRU"""
    print("\nIdentificando pazienti estremi dal test set GRU...")

    results_df = pd.read_csv(gru_results_file)

    # Calcola metriche per paziente
    patient_metrics = []
    for patient_id in results_df["Patient_ID"].unique():
        patient_data = results_df[results_df["Patient_ID"] == patient_id]
        samples, maes, mapes, rmses = calculate_metrics(patient_data)
        patient_metrics.append(
            {
                "Patient_ID": patient_id,
                "Samples": samples,
                "MAE": maes[0] if len(maes) > 0 else np.nan,
                "MAPE": mapes[0] if len(mapes) > 0 else np.nan,
                "RMSE": rmses[0] if len(rmses) > 0 else np.nan,
            }
        )

    patient_metrics_df = pd.DataFrame(patient_metrics).dropna()

    # Identifica estremi basati su MAE
    best_patient_id = patient_metrics_df.loc[
        patient_metrics_df["MAE"].idxmin(), "Patient_ID"
    ]
    worst_patient_id = patient_metrics_df.loc[
        patient_metrics_df["MAE"].idxmax(), "Patient_ID"
    ]

    # Ottieni le metriche per il migliore e peggiore paziente
    best_patient_metrics = patient_metrics_df[
        patient_metrics_df["Patient_ID"] == best_patient_id
    ].iloc[0]
    worst_patient_metrics = patient_metrics_df[
        patient_metrics_df["Patient_ID"] == worst_patient_id
    ].iloc[0]

    print(f"Migliore paziente: {best_patient_id}")
    print(f"  MAE: {best_patient_metrics['MAE']:.2f} mg/dL")
    print(f"  MAPE: {best_patient_metrics['MAPE']:.2f}%")
    print(f"  RMSE: {best_patient_metrics['RMSE']:.2f} mg/dL")
    print(f"  Campioni: {best_patient_metrics['Samples']}")

    print(f"Peggiore paziente: {worst_patient_id}")
    print(f"  MAE: {worst_patient_metrics['MAE']:.2f} mg/dL")
    print(f"  MAPE: {worst_patient_metrics['MAPE']:.2f}%")
    print(f"  RMSE: {worst_patient_metrics['RMSE']:.2f} mg/dL")
    print(f"  Campioni: {worst_patient_metrics['Samples']}")

    return int(best_patient_id), int(worst_patient_id)

In [ ]:
def load_original_data():
    """Carica i dati originali completi"""
    print("\nCaricando dati originali...")

    # Carica splits originali per ottenere X_cols e y_cols
    train_set, val_set, test_set, X_cols, y_cols = load_splits()

    # Combina tutti i dati per avere il dataset completo
    all_data = pd.concat([train_set, val_set, test_set], ignore_index=True)
    all_data = all_data.sort_values(["Patient_ID", "Timestamp"]).reset_index(drop=True)

    print(f"Caricati {len(all_data)} campioni totali")

    return all_data, X_cols, y_cols

In [ ]:
def create_patient_splits(patient_data, test_split, patient_id):
    """Crea split temporali per un singolo paziente (80% train, 20% test)"""
    # Ordina per timestamp per mantenere ordine temporale
    patient_data = patient_data.sort_values("Timestamp").reset_index(drop=True)

    # Split temporale: primi 80% per train, ultimi 20% per test
    split_idx = int(len(patient_data) * (1 - test_split))

    train_data = patient_data.iloc[:split_idx].copy()
    test_data = patient_data.iloc[split_idx:].copy()

    print(f"  Paziente {patient_id}: {len(train_data)} train, {len(test_data)} test")

    return train_data, test_data

In [ ]:
def train_personalized_xgb(train_data, X_cols, y_cols, patient_id, args):
    """Addestra XGBoost personalizzato per un paziente"""
    print(f"\nAddestrando XGBoost personalizzato per paziente {patient_id}...")

    # Carica parametri ottimizzati da Optuna
    optuna_study_path = "tuning/results/xgb_optuna_study.pkl"

    try:
        with open(optuna_study_path, "rb") as f:
            study = pickle.load(f)
        best_params = study.best_params

        # Usa i parametri ottimizzati
        print(f"  Usando parametri ottimizzati: {best_params}")

    except FileNotFoundError:
        print(f"  File {optuna_study_path} non trovato")

    model = xgb.XGBRegressor(
        **best_params,
        random_state=args.seed,
        device="cuda:0",
    )

    model.fit(
        train_data[X_cols],
        train_data[y_cols[-1]],
    )

    # Salva modello
    model_path = f"{args.models_dir}/xgb_patient_{patient_id}.pickle"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    print(f"Modello XGBoost personalizzato salvato: {model_path}")

    return model

In [ ]:
def train_personalized_gru(train_data, test_data, X_cols, y_cols, patient_id, args):
    """Addestra GRU personalizzato per un paziente"""
    print(f"\nAddestrando GRU personalizzato per paziente {patient_id}...")

    # Prepara dati per GRU (usa il train-set per l'addestramento e il test-set per l'early stopping)
    X_train = train_data[X_cols].values.reshape(len(train_data), len(X_cols), 1)
    y_train = train_data[y_cols[-1]].values

    X_test = test_data[X_cols].values.reshape(len(test_data), len(X_cols), 1)
    y_test = test_data[y_cols[-1]].values

    print(f"  Forma dati training: X_train {X_train.shape}, y_train {y_train.shape}")
    print(f"  Forma dati testing: X_test {X_test.shape}, y_test {y_test.shape}")

    # Crea modello GRU con architettura ottimale
    print(f"\nCreating personalized GRU model...")

    set_seeds(args.seed)

    model = create_gru_model()

    print_model_summary(model)

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=args.lr),
        loss=keras.losses.Huber(),
        metrics=["mae"],
        steps_per_execution=256,
    )

    callbacks = create_callbacks(
        early_stopping_patience=args.es_patience, early_stopping_min_delta=0
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        verbose=1,
    )

    # Salva i pesi del miglior modello alla fine del training
    # (EarlyStopping con restore_best_weights=True già ripristina i migliori pesi)
    model_save_path = f"{args.models_dir}/gru_patient_{patient_id}.weights.h5"
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    model.save_weights(model_save_path)
    print(f"Best model weights saved to: {model_save_path}")

    return model

In [ ]:
def evaluate_personalized_models(
    patient_id, test_data, X_cols, y_cols, xgb_model, gru_model, args
):
    """Valuta modelli personalizzati su dati test del paziente"""
    print(f"\nValutando modelli personalizzati per paziente {patient_id}...")

    # Predizioni XGBoost personalizzato
    xgb_pred = xgb_model.predict(test_data[X_cols])

    # Predizioni GRU personalizzato
    gru_pred = predict_in_batches(gru_model, test_data[X_cols], "gru")

    # Crea DataFrame risultati
    results = test_data.copy()
    results["target"] = test_data[y_cols[-1]]
    results["xgb_personalized"] = xgb_pred
    results["gru_personalized"] = gru_pred

    # Rescale ai valori originali
    results = rescale_data(results, ["target", "xgb_personalized", "gru_personalized"])

    # Salva risultati
    output_file = f"{args.output_dir}/xgb_and_gru_output_patient_{patient_id}.csv"
    results[
        [
            "Timestamp",
            "Patient_ID",
            "bgClass",
            "target",
            "xgb_personalized",
            "gru_personalized",
        ]
    ].to_csv(output_file, index=False)

    print(f"Risultati personalizzati salvati: {output_file}")

    return results

In [ ]:
def load_generalized_results(patient_id, xgb_results_file, gru_results_file):
    """Carica risultati modelli generalizzati per un paziente specifico"""
    print(f"\nCaricando risultati generalizzati per paziente {patient_id}...")

    # Carica risultati XGBoost generalizzato
    xgb_results = pd.read_csv(xgb_results_file)
    xgb_patient = xgb_results[xgb_results["Patient_ID"] == patient_id].copy()

    # Carica risultati GRU generalizzato
    gru_results = pd.read_csv(gru_results_file)
    gru_patient = gru_results[gru_results["Patient_ID"] == patient_id].copy()

    # Crea dataset combinato
    generalized_results = xgb_patient[
        ["Timestamp", "Patient_ID", "bgClass", "target"]
    ].copy()
    generalized_results["xgb_generalized"] = xgb_patient["y_pred"].values
    generalized_results["gru_generalized"] = gru_patient["y_pred"].values

    print(f"Caricati {len(generalized_results)} risultati generalizzati")

    return generalized_results

In [ ]:
def calculate_comparison_metrics(
    patient_id, personalized_results, generalized_results, args
):
    """Calcola metriche comparative tra modelli generalizzati e personalizzati"""
    print(f"\nCalcolando metriche comparative per paziente {patient_id}...")

    # Usa il dataset personalizzato come riferimento (ultimi 20% dei dati)
    # e trova i corrispondenti nel dataset generalizzato basandosi sull'ordine temporale

    print(f"  Personalizzato: {len(personalized_results)} campioni")
    print(f"  Generalizzato: {len(generalized_results)} campioni")

    # Prendi gli ultimi N campioni dal dataset generalizzato dove N = len(personalized_results)
    n_samples = len(personalized_results)
    generalized_subset = generalized_results.tail(n_samples).reset_index(drop=True)
    personalized_subset = personalized_results.reset_index(drop=True)

    # Usa target dal dataset personalizzato
    target = personalized_subset["target"].values

    print(f"  {len(target)} campioni per confronto")

    # Calcola metriche per tutti i modelli
    metrics_data = []

    models = {
        "XGBoost_Generalized": generalized_subset["xgb_generalized"].values,
        "XGBoost_Personalized": personalized_subset["xgb_personalized"].values,
        "GRU_Generalized": generalized_subset["gru_generalized"].values,
        "GRU_Personalized": personalized_subset["gru_personalized"].values,
    }

    for model_name, predictions in models.items():
        mae = mean_absolute_error(target, predictions)
        mape = mean_absolute_percentage_error(target, predictions)
        rmse = root_mean_squared_error(target, predictions)

        # Clarke Error Grid
        ceg_stats = clarke_error_grid_analysis(
            target,
            predictions,
            f"{model_name} - Patient {patient_id}",
            save_path=f"{args.plots_dir}/ceg_{model_name.lower()}_patient_{patient_id}.png",
        )

        metrics_data.append(
            {
                "Patient_ID": patient_id,
                "Model": model_name,
                "Samples": len(target),
                "MAE": mae,
                "MAPE": mape,
                "RMSE": rmse,
                "Zone_A_Pct": ceg_stats["zone_percentages"]["A"],
                "Zone_B_Pct": ceg_stats["zone_percentages"]["B"],
                "Zone_C_Pct": ceg_stats["zone_percentages"]["C"],
                "Zone_D_Pct": ceg_stats["zone_percentages"]["D"],
                "Zone_E_Pct": ceg_stats["zone_percentages"]["E"],
                "Clinically_Acceptable": ceg_stats["clinically_acceptable"],
                "Clinically_Dangerous": ceg_stats["clinically_dangerous"],
            }
        )

    return pd.DataFrame(metrics_data)

In [ ]:
def create_comparison_summary(all_metrics_df, args):
    """Crea riassunto finale del confronto"""
    print("\nCreando riassunto finale del confronto...")

    # Salva metriche complete
    metrics_file = f"{args.scores_dir}/complete_comparison.csv"
    all_metrics_df.to_csv(metrics_file, index=False)
    print(f"Metriche complete salvate: {metrics_file}")

    # Crea tabella riassuntiva
    summary_data = []

    for patient_id in all_metrics_df["Patient_ID"].unique():
        patient_metrics = all_metrics_df[all_metrics_df["Patient_ID"] == patient_id]

        for metric in ["MAE", "MAPE", "RMSE", "Clinically_Acceptable"]:
            row = {"Patient_ID": patient_id, "Metric": metric}

            for model in [
                "XGBoost_Generalized",
                "XGBoost_Personalized",
                "GRU_Generalized",
                "GRU_Personalized",
            ]:
                value = patient_metrics[patient_metrics["Model"] == model][
                    metric
                ].values[0]
                if metric in ["MAE", "RMSE"]:
                    row[model] = f"{value:.2f}"
                elif metric == "MAPE":
                    row[model] = f"{value:.2f}%"
                else:  # Clinically_Acceptable
                    row[model] = f"{value:.2f}%"

            summary_data.append(row)

    summary_df = pd.DataFrame(summary_data)
    # summary_file = f"{output_dir}/comparison_summary.csv"
    # summary_df.to_csv(summary_file, index=False)
    # print(f"Riassunto salvato: {summary_file}")

    # Stampa riassunto
    print("\n" + "=" * 80)
    print("CONFRONTO MODELLI GENERALIZZATI VS PERSONALIZZATI")
    print("=" * 80)

    for patient_id in sorted(all_metrics_df["Patient_ID"].unique()):
        print(f"\nPAZIENTE {patient_id}:")
        patient_data = summary_df[summary_df["Patient_ID"] == patient_id]

        for _, row in patient_data.iterrows():
            print(
                f"  {row['Metric']:<20}: XGB_Gen {row['XGBoost_Generalized']:<8} | XGB_Pers {row['XGBoost_Personalized']:<8} | GRU_Gen {row['GRU_Generalized']:<8} | GRU_Pers {row['GRU_Personalized']:<8}"
            )

    print("\n" + "=" * 80)

    return summary_df

### Main pipeline


In [ ]:
# Configurazione
class Args:
    def __init__(self):
        self.gru_results_file = "outputs/test_set/gru_output.csv"
        self.xgb_results_file = "outputs/test_set/xgb_output.csv"
        self.output_dir = "outputs/personalized"
        self.models_dir = "models/personalized"
        self.scores_dir = "scores/gen_vs_pers"
        self.plots_dir = "plots/gen_vs_pers"
        self.test_split = 0.2
        self.seed = 42
        self.epochs = 100
        self.batch_size = 256
        self.lr = 0.01
        self.es_patience = 20


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - GRU results file: {args.gru_results_file}")
print(f"  - XGB results file: {args.xgb_results_file}")
print(f"  - Output dir: {args.output_dir}")
print(f"  - Models dir: {args.models_dir}")
print(f"  - Scores dir: {args.scores_dir}")
print(f"  - Plots dir: {args.plots_dir}")
print(f"  - Test split: {args.test_split}")
print(f"  - Seed: {args.seed}")
print(f"  - Epochs: {args.epochs}")
print(f"  - Batch size: {args.batch_size}")
print(f"  - Learning rate: {args.lr}")
print(f"  - Early stopping patience: {args.es_patience}")

In [ ]:
print("CONFRONTO MODELLI GENERALIZZATI VS PERSONALIZZATI")
print("=" * 60)

# Setup environment
setup_environment(args)

In [ ]:
# Identifica i pazienti estremi
best_patient, worst_patient = identify_extreme_patients(args.gru_results_file)
target_patients = [best_patient, worst_patient]

print(f"Target patients: {target_patients}")

In [ ]:
# Carica dati originali
all_data, X_cols, y_cols = load_original_data()

print(f"Features: {len(X_cols)} columns")
print(f"Target: {y_cols}")

In [ ]:
# Raccoglitore per tutte le metriche
all_metrics = []

# Processa ogni paziente target
for patient_id in target_patients:
    print(f"\n" + "=" * 50)
    print(f"PROCESSING PAZIENTE {patient_id}")
    print("=" * 50)

    # Estrai dati del paziente
    patient_data = all_data[all_data["Patient_ID"] == patient_id].copy()
    print(f"Dati totali paziente {patient_id}: {len(patient_data)} campioni")

    # Crea split temporali per addestramento modelli personalizzati
    train_data, test_data = create_patient_splits(
        patient_data, args.test_split, patient_id
    )

    # Addestra modelli personalizzati
    xgb_personalized = train_personalized_xgb(
        train_data, X_cols, y_cols, patient_id, args
    )
    gru_personalized = train_personalized_gru(
        train_data, test_data, X_cols, y_cols, patient_id, args
    )

    # Valuta modelli personalizzati
    personalized_results = evaluate_personalized_models(
        patient_id,
        test_data,
        X_cols,
        y_cols,
        xgb_personalized,
        gru_personalized,
        args,
    )

    # Carica risultati modelli generalizzati
    generalized_results = load_generalized_results(
        patient_id, args.xgb_results_file, args.gru_results_file
    )

    # Calcola metriche comparative
    patient_metrics = calculate_comparison_metrics(
        patient_id, personalized_results, generalized_results, args
    )

    if patient_metrics is not None:
        all_metrics.append(patient_metrics)

    print(f"Completato processamento paziente {patient_id}")

In [ ]:
# Combina tutte le metriche
if all_metrics:
    all_metrics_df = pd.concat(all_metrics, ignore_index=True)

    # Crea riassunto finale
    summary_df = create_comparison_summary(all_metrics_df, args)

    print(f"\nConfronto completato! Risultati salvati in: {args.output_dir}")

    results = {
        "metrics": all_metrics_df,
        "summary": summary_df,
        "target_patients": target_patients,
    }
else:
    print("\nNessun risultato generato")
    results = None